# Notes:

Metrics to add:
- column for type of workout so we can predict in different workouts


Time sequence options:
- pad at the end of each sequence and build an LSTM RNN with a layer that ignores padded portions of the sequence and a bidirectional layer to retain context from each phase
    1) separate pitch into phases (wind up, sit down power, and throw maybe? or possibly: wind-up, stride, arm cocking, acceleration, release)
        - ENSURE phases are always started the same, this takes away the noise of different starts and is ESSENTIAL or just do tree based instead of lstm
        - Add phase ID (e.g., wind_up=0, acceleration=1, release=2) as a feature to the model.
    2) use time column to split into different sequences (per trial)
        - consider normalizing the time sequences to 0,1. This only makes sense if it's not already normalized, check first and check distance in time steps to ensure no noise
    3) pad at the end of each phase so that it each sequence matches the max sequence
        - Pad each phase separately to its own max length (e.g., wind-up padded to 20 steps, acceleration to 50 steps).
        - Use a hierarchical model (e.g., separate LSTM per phase) to avoid mixing padded values across phases.
    4) create masked neural network that is bidirectional for edge cases and masking so we exclude the padded steps
        For causal tasks, use unidirectional LSTMs.
        For non-causal tasks (e.g., post-pitch classification), use bidirectional layers.
            from tensorflow.keras.layers import Input, Masking, Bidirectional, LSTM, Dense

            # Phase-specific input (e.g., wind-up)
            inputs = Input(shape=(max_phase_length, num_features + 2))  # +2 for phase ID and normalized time
            masked = Masking(mask_value=-1.0)(inputs)  # Assume padded time = -1
            lstm_out = Bidirectional(LSTM(64))(masked)  
            outputs = Dense(1, activation='sigmoid')(lstm_out)
        Add attention mechanisms to explicitly highlight critical moments (e.g., release):
            python
            from tensorflow.keras.layers import Attention, Concatenate
            # Add attention to focus on key timesteps (e.g., release)
            context_vector = Attention()([lstm_out, lstm_out])
            outputs = Concatenate()([lstm_out, context_vector])


Other options to test
- Dynamic Time Warping (DTW): Align sequences non-linearly to a reference length.
    Use Case: Best for comparing shapes of biomechanical curves (e.g., elbow angle vs. time).
    Use DTW as a preprocessing step for similarity-based tasks (e.g., clustering pitches by motion pattern).

- Resampling: Interpolate sequences to a fixed length (e.g., 100 steps) using biomechanical curves’ shape.
    Use Case: Suitable when biomechanical patterns are time-invariant (e.g., joint angles during release).
    Use cubic spline interpolation instead of linear to preserve curve shape.


Key Tests to Validate Approach

    Phase Alignment Check:

        Plot the distribution of phase durations (e.g., wind-up: 0.2–0.3s, release: 0.1–0.15s).

        Ensure padding doesn’t exceed 20% of the phase’s natural duration.
Key Tests to Validate Approach

    Phase Alignment Check:

        Plot the distribution of phase durations (e.g., wind-up: 0.2–0.3s, release: 0.1–0.15s).

        Ensure padding doesn’t exceed 20% of the phase’s natural duration.

    Ablation Study:

        Compare performance of:
        a) Phase-padded LSTM
        b) Resampled LSTM
        c) DTW-aligned model

    Attention Visualization:

        Use libraries like tf-explain to confirm attention focuses on biomechanically critical moments (e.g., release).

    Noise Injection Test:

        Add Gaussian noise to padded regions and verify model robustness (accuracy shouldn’t degrade)
    Ablation Study:

        Compare performance of:
        a) Phase-padded LSTM
        b) Resampled LSTM
        c) DTW-aligned model

    Attention Visualization:

        Use libraries like tf-explain to confirm attention focuses on biomechanically critical moments (e.g., release).

    Noise Injection Test:

        Add Gaussian noise to padded regions and verify model robustness (accuracy shouldn’t degrade)

# Trials analysis

# Merge with EMG metrics
- Load From:
    * emg_data_parse_and_load_final.ipynb validates and parses the emg data with the timestamp added on, will add on bulk loading and kafka datastreaming so it's better for this case
Things to watch out for:
Tolerance in join difference? should it be within .0001 sec or how big would we set up tolerance to match up those different systems?
Validation to ensure we aren't losing any metrics from either


possibly this:
Alternative Approach: Using merge_asof for Nearest-Time Joins

If the datasets have timestamps that are close but not identical and you’d prefer to merge based on the nearest time (within a certain tolerance), you can use pd.merge_asof.

# Sort dataframes by datetime
df1 = df1.sort_values('datetime')
df2 = df2.sort_values('datetime')

# Perform an asof merge: This will match each row in df1 to the nearest row in df2
merged_asof = pd.merge_asof(df1, df2, on='datetime', direction='nearest', tolerance=pd.Timedelta('1s'))

Note: With merge_asof, it’s important that the data is sorted by the datetime column, and you can set a tolerance (e.g., one second) to control how far apart the matching timestamps can be.

In [12]:
import pandas as pd
import numpy as np

# -------------------------------
# Step 1: Load and Inspect DataFrames
# -------------------------------
# Load the processed biomechanical (granular joint) data.
merged_df_left = pd.read_parquet('../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet')
print("merged_df_left columns:")
print(merged_df_left.columns)
print("\nmerged_df_left sample data:")
print(merged_df_left.head())

# Load the processed EMG data.
EMG_metrics = pd.read_csv('../../data/processed/processed_pitch_data.csv')
print("\nEMG_metrics columns:")
print(EMG_metrics.columns)
print("\nEMG_metrics sample data:")
print(EMG_metrics.head())

# -------------------------------
# Step 2: Prepare Datetime Columns for Biomech Data
# -------------------------------
if 'ongoing_timestamp' in merged_df_left.columns:
    merged_df_left['ongoing_timestamp'] = pd.to_datetime(merged_df_left['ongoing_timestamp'])
    merged_df_left['datetime'] = merged_df_left['ongoing_timestamp']
else:
    raise ValueError("merged_df_left does not contain a suitable datetime column (e.g., 'ongoing_timestamp').")

print("\nBase dataset datetime sample:")
print(merged_df_left[['ongoing_timestamp', 'datetime']].head())

# -------------------------------
# Step 3: Prepare Datetime Columns in EMG_metrics
# -------------------------------
if 'Date/Time' in EMG_metrics.columns and 'Timestamp' in EMG_metrics.columns:
    EMG_metrics['Date/Time_parsed'] = pd.to_datetime(EMG_metrics['Date/Time'])
    EMG_metrics['Timestamp_parsed'] = pd.to_datetime(EMG_metrics['Timestamp'])
    # Use the parsed Timestamp as the common datetime field.
    EMG_metrics['emg_time'] = EMG_metrics['Timestamp']
    EMG_metrics['datetime'] = EMG_metrics['Timestamp_parsed']
elif 'Date/Time' in EMG_metrics.columns:
    EMG_metrics['emg_time'] = EMG_metrics['Date/Time']
    EMG_metrics['datetime'] = pd.to_datetime(EMG_metrics['Date/Time'])
elif 'Timestamp' in EMG_metrics.columns:
    EMG_metrics['emg_time'] = EMG_metrics['Timestamp']
    EMG_metrics['datetime'] = pd.to_datetime(EMG_metrics['Timestamp'])
else:
    raise ValueError("EMG_metrics does not contain a suitable datetime column (e.g., 'Date/Time' or 'Timestamp').")

print("\nmerged_df_left datetime range:", merged_df_left['datetime'].min(), "to", merged_df_left['datetime'].max())
print("EMG_metrics datetime range:", EMG_metrics['datetime'].min(), "to", EMG_metrics['datetime'].max())
print("EMG_metrics Timestamp range:", EMG_metrics['Timestamp'].min(), "to", EMG_metrics['Timestamp'].max())

# -------------------------------
# Step 4: Compute Time Steps and Sort DataFrames
# -------------------------------
merged_df_left['time_step'] = merged_df_left['datetime'].diff()
print("\nTime step differences in merged_df_left (first 10 rows):")
print(merged_df_left[['datetime', 'time_step']].head(10))

merged_df_left = merged_df_left.sort_values('datetime')
EMG_metrics = EMG_metrics.sort_values('datetime')

print("\nRow count of merged_df_left:", merged_df_left.shape[0])
print("Row count of EMG_metrics:", EMG_metrics.shape[0])

# -------------------------------
# Step 5: Inner Join of Biomech and EMG Data
# -------------------------------
# Using merge_asof to align rows within a 10ms tolerance.
# Note: merge_asof performs a left join, so we then drop rows with no EMG match.
merged_inner = pd.merge_asof(
    merged_df_left,
    EMG_metrics,
    on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta('10ms')
)

# Drop rows where no matching EMG record was found (using a key column from EMG_metrics, e.g., 'Timestamp').
merged_inner = merged_inner.dropna(subset=['Timestamp'])

print("\nAfter inner join between biomech and EMG data:")
print("Row count:", merged_inner.shape[0])
print(merged_inner.head())

# -------------------------------
# Step 6: Save the Final Inner Join Dataset
# -------------------------------
output_inner_path = '../../data/processed/ml_datasets/merged_biomech_emg_inner_join_data.parquet'
merged_inner.to_parquet(output_inner_path, index=False)
print(f"\nInner join dataset saved to: {output_inner_path}")

# -------------------------------
# Step 7: Left Join of Biomech Data onto EMG_metrics (EMG Base Metrics)
# -------------------------------
# Here, we use EMG_metrics as the base and left join the biomech data from merged_df_left
# within the same 10ms tolerance.
emg_base_metrics = pd.merge_asof(
    EMG_metrics,
    merged_df_left,
    on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta('10ms')
)

print("\nAfter left join of biomech data onto EMG metrics (emg_base_metrics):")
print("Row count:", emg_base_metrics.shape[0])
print(emg_base_metrics.head())

# -------------------------------
# Step 8: Save the EMG Base Metrics Dataset
# -------------------------------
output_emg_base_path = '../../data/processed/ml_datasets/merged_emg_base_metrics_data.parquet'
emg_base_metrics.to_parquet(output_emg_base_path, index=False)
print(f"\nEMG base metrics dataset saved to: {output_emg_base_path}")

# -------------------------------
# Step 9: Deep Analysis on the Joins
# -------------------------------
print("\n=== Deep Analysis on the Inner Join Dataset ===")
print("Inner join dataset shape:", merged_inner.shape)
print("\nNull counts in inner join dataset:")
print(merged_inner.isnull().sum())

# Check for duplicate datetime rows (which might indicate an 'explosion' in the join)
inner_duplicates = merged_inner.duplicated(subset=['datetime']).sum()
print("\nNumber of duplicate 'datetime' rows in inner join dataset:", inner_duplicates)

# Compare the inner join to the original biomech dataset
biomech_unmatched = merged_df_left.shape[0] - merged_inner.shape[0]
print("\nNumber of biomech rows with no matching EMG (dropped in inner join):", biomech_unmatched)

print("\n=== Deep Analysis on the EMG Base Metrics Dataset (Left Join) ===")
print("EMG base metrics dataset shape:", emg_base_metrics.shape)
print("\nNull counts in EMG base metrics dataset:")
print(emg_base_metrics.isnull().sum())

# For EMG base metrics, check how many rows did NOT match any biomech record.
# Assuming 'ongoing_timestamp' comes only from the biomech data.
emg_unmatched = emg_base_metrics['ongoing_timestamp'].isnull().sum()
print("\nNumber of EMG rows with no matching biomech data:", emg_unmatched)

# Check for duplicate datetime rows here as well
emg_duplicates = emg_base_metrics.duplicated(subset=['datetime']).sum()
print("\nNumber of duplicate 'datetime' rows in EMG base metrics dataset:", emg_duplicates)


merged_df_left columns:
Index(['athlete_name', 'athlete_dob', 'athlete_traq', 'height_meters',
       'mass_kilograms', 'athlete_level', 'session_date', 'session_time',
       'lab', 'session', 'trial', 'pitch_type', 'handedness',
       'ongoing_timestamp', 'session_trial', 'time', 'shoulder_angle_x',
       'shoulder_angle_y', 'shoulder_angle_z', 'elbow_angle_x',
       'elbow_angle_y', 'elbow_angle_z', 'torso_angle_x', 'torso_angle_y',
       'torso_angle_z', 'pelvis_angle_x', 'pelvis_angle_y', 'pelvis_angle_z',
       'shoulder_velo_x', 'shoulder_velo_y', 'shoulder_velo_z', 'elbow_velo_x',
       'elbow_velo_y', 'elbow_velo_z', 'torso_velo_x', 'torso_velo_y',
       'torso_velo_z', 'trunk_pelvis_dissociation', 'pitch_phase',
       'shoulder_energy_transfer', 'shoulder_energy_generation',
       'elbow_energy_transfer', 'elbow_energy_generation',
       'lead_knee_energy_transfer', 'lead_knee_energy_generation',
       'elbow_moment_x', 'elbow_moment_y', 'elbow_moment_z',
       's

EDA
Check all the basics:
- p value for each metric for suspicious stuff
- outliers
- standardization
- normalization
- dispersion
- disparity
- nulls and which to take out or to impute
- anything else that fixes this dataset 

# Feature Engineering

metrics to add:
* Set up phases based on the pitcher movement so we can set them into different phases
* add the phases to the EMG dataset so we can better predict on that and create use the merge_asof dataset to see if it's worth predicting with for the time series motion of the joints (the merge_asof would need to be a small difference)
* add in pulse data if we don't have arm slot/torque/speed in the database
* add in vulgas range and possibly the ongoing detriment from that
* finish up the sql statement (ensure it's completely validated) for the base table
* separate the dataset into time series classification, regression and tree based classification datasets
* check the feaures importances and such
* use permutation importance and use github(dot)com/eli5-org/eli5

Setting up feature dictionary to speak over each feature to talk about actual important metrics outside of those feature selected
Feature Selection (RFE, SHAP importance, correlation (to take out multi-collinearity metrics or combine them))
